In [2]:
import pandas as pd
import sqlite3
import os
import re

# --- 1. Hitta och anslut till databasen ---
db_path = os.path.abspath(os.path.join(os.getcwd(), "..", "db", "database.db"))
print(f"Använder databas: {db_path}")

conn = sqlite3.connect(db_path)

# --- 2. Läs tabeller dynamiskt ---
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Tabeller i databasen:", tables['name'].tolist())

dfs = {}
for table in tables['name']:
    dfs[table] = pd.read_sql_query(f"SELECT * FROM {table}", conn)

# Tilldela för enkelhet
transactions = dfs.get('transactions', pd.DataFrame())
accounts = dfs.get('accounts', pd.DataFrame())
customers = dfs.get('customers', pd.DataFrame())

# --- 3. Slå ihop data dynamiskt ---
df = transactions.copy()
if not accounts.empty and 'account_id' in df.columns and 'id' in accounts.columns:
    df = df.merge(accounts, left_on='account_id', right_on='id', suffixes=("", "_account"))

if 'customer_id' in df.columns and not customers.empty and 'id' in customers.columns:
    df = df.merge(customers, left_on='customer_id', right_on='id', suffixes=("", "_customer"))

print(f"Antal rader efter merge: {len(df)}")
display(df.head())

# --- 4. Valideringar ---

# 4.1 Saknade värden
missing_values = df.isnull().sum()
print("\nSaknade värden per kolumn:")
print(missing_values[missing_values > 0])

# 4.2 Kolla datumformat
date_cols = [col for col in df.columns if "date" in col.lower()]
for col in date_cols:
    try:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    except Exception as e:
        print(f"Fel vid konvertering av {col}: {e}")
bad_dates = {col: df[df[col].isna()][col] for col in date_cols if df[col].isna().any()}
if bad_dates:
    print("\nFelaktiga datumvärden:")
    for col, vals in bad_dates.items():
        print(f"- {col}: {len(vals)} fel")

# 4.3 Kolla IBAN (grundläggande kontroll)
iban_cols = [col for col in df.columns if "iban" in col.lower() or "account_number" in col.lower()]
iban_pattern = re.compile(r'^[A-Z]{2}[0-9]{2}[A-Z0-9]{1,30}$')
invalid_ibans = {}
for col in iban_cols:
    invalid = df[~df[col].astype(str).str.match(iban_pattern, na=False)]
    if not invalid.empty:
        invalid_ibans[col] = invalid[col]
if invalid_ibans:
    print("\nFelaktiga IBAN/Account-nummer:")
    for col, vals in invalid_ibans.items():
        print(f"- {col}: {len(vals)} fel")

# 4.4 Negativa transaktionsbelopp
if 'amount' in df.columns:
    negative_transactions = df[df['amount'] < 0]
    print(f"\nAntal negativa transaktioner: {len(negative_transactions)}")
    display(negative_transactions)

# 4.5 Misstänkta transaktioner (belopp > 100000 eller destination okänt land)
suspicious = pd.DataFrame()
if 'amount' in df.columns:
    suspicious = df[df['amount'] > 100000]
if 'country' in df.columns:
    suspicious = pd.concat([suspicious, df[df['country'].isna() | (df['country'] == '')]]).drop_duplicates()

if not suspicious.empty:
    print(f"\nMisstänkta transaktioner: {len(suspicious)}")
    display(suspicious)

conn.close()


Använder databas: c:\Users\stenm\Desktop\Data Manager\Datakvalitet\Projektarbete\bank-project\db\database.db
Tabeller i databasen: ['customers', 'transactions']
Antal rader efter merge: 100000


,transaction_id,timestamp,amount,currency,sender_account,receiver_account,sender_country,sender_municipality,receiver_country,receiver_municipality,transaction_type,notes
0,62cacc89-95ca-41c0-a06f-d12950c85a37,2025-01-08 03:17:00,1174.47,SEK,SE8902ZCTK81497780781136,SE8902GGWE97497502378359,Sweden,Södertälje,Sweden,Stockholm,outgoing,Tax refund
1,d1ae6818-9d0f-4e14-bc56-0165e8123ff0,2025-01-02 19:34:00,19068.02,SEK,SE8902DGXX88147139459340,SE8902PLBL16144667744762,Sweden,Uddevalla,Sweden,Karlskoga,incoming,Salary payment
2,aa707bbc-7171-4a8f-bdd1-5a18943708d5,2025-01-12 20:08:00,7557.86,SEK,SE8902VMWP60739361675669,SE8902JTIU30473679174736,Sweden,Gävle,Sweden,Nyköping,outgoing,Subscription fee
3,40adfe97-ed7c-483e-bd50-cdec4726fc80,2025-02-08 06:24:00,14108.32,SEK,SE8902GHSB29020078073922,SE8902MQTA07583386361028,Sweden,Trelleborg,Sweden,Gävle,outgoing,Payment for invoice #1934
4,b078f054-3ce8-4d2a-9173-6e0c94a9bbff,2025-03-01 18:51:00,33570.37,SEK,SE8902NHEC01255630352421,SE8902HQID54921038055598,Sweden,Borlänge,Sweden,Landskrona,outgoing,None



Saknade värden per kolumn:
sender_country            500
sender_municipality       500
receiver_country          500
receiver_municipality     500
notes                    9948
dtype: int64

Antal negativa transaktioner: 0


,transaction_id,timestamp,amount,currency,sender_account,receiver_account,sender_country,sender_municipality,receiver_country,receiver_municipality,transaction_type,notes
